# OpenPlaque — LCX Structural Identity Adjudication
Fresh-baseline, gate-based adjudication of already validated C6/C7 evidence. No new vessel search. Single-kernel execution.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR = DRIVE_ROOT + '/LCX_Structural_Identity_Adjudication_v1'
BRANCH = 'lcx-structural-identity-adjudication-from-main'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
print('Branch:', BRANCH)
print('Output:', OUTPUT_DIR)


In [ ]:
import os, shutil
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone --depth 1 --branch $BRANCH https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
HEAD = !git -C /content/OpenPlaque rev-parse HEAD
HEAD = HEAD[0].strip()
print('Checked out HEAD:', HEAD)


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy


In [ ]:
import sys, importlib, pytest
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import lcx_structural_identity_adjudication as exp
print('openplaque:', openplaque.__file__)
print('algorithm:', exp.ALGORITHM)
assert exp.BASELINE == BASELINE
assert exp.ALGORITHM == 'lcx-structural-identity-adjudication-v1.0'
print('self-test:', exp.synthetic_adjudication_self_test())
rc = pytest.main(['-q','/content/OpenPlaque/tests/test_lcx_structural_identity_adjudication.py'])
if rc != 0: raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries/coronary_arteries.nii.gz',
 root/'TotalSegmentator_Cardiovascular_Cache_v1/coronary_arteries_LEGACY/coronary_arteries.nii.gz',
 root/'Joint_Three_Vessel_Template_Classifier_v1/candidate_04_source_path.csv',
 root/'LCX_Consensus_Trunk_AV_Groove_v1/summary.json',
 root/'LCX_Parent_Continuation_Topology_v1/summary.json',
 root/'LCX_Chamber_Interface_Midpoint_v1/summary.json',
 root/'LCX_Distal_Reacquisition_v1_fixed/summary.json',
 root/'LCX_Distal_Reacquisition_v1_fixed/C7_extended_path.csv',
 root/'LCX_C6_Monotonic_Distal_v1/summary.json',
]
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))


In [ ]:
import gc
from openplaque.lcx_structural_identity_adjudication import run
gc.collect()
result = run(DRIVE_ROOT, OUTPUT_DIR)
print('STATUS:', result['summary']['status'])
print('GATES:', result['summary']['gates'])
print('DECISION:', result['summary']['decision'])
print('REPORT:', result['report'])
print('ZIP:', result['zip'])
